> **Original course-project notebook (2026), kept for reference; superseded by `src/dfer/`.** Outputs are cleared: they contained DFEW frames (licence-restricted) and results that were selected on the fold's test set and affected by a frame-ordering bug. See the README for the corrected pipeline and results.

# Qualitative Evaluation of Video Emotion Recognition (DFEW)

This notebook performs qualitative evaluation of a trained VideoMAE model on the DFEW dataset.

The main objectives are:
- Load the test split of the dataset
- Run inference using a trained model
- Compare predictions with ground truth labels
- Visualize correct and incorrect predictions

This allows us to better understand the behavior of the model beyond quantitative metrics such as accuracy or UAR.

## Import Required Libraries

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import VideoMAEForVideoClassification

## Define Paths

In [ ]:
clip_root = "data/DEFW Dataset/Clip/clip_224x224_16f"

val_csv = "data/DEFW Dataset/EmoLabel_DataSplit/test(single-labeled)/set_1.csv"

checkpoint_path = "data/dfew_ssl_project/outputs/checkpoints/02_supervised_vit_baseline_fold1_FINAL_best_uar.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load Test Split

We load the test split from the DFEW dataset using a CSV file.

The CSV contains:
- `video_name`: identifier of each video
- `label`: ground truth emotion class

We extract:
- A list of video IDs
- A list of corresponding labels

We also format video IDs (zero-padding) to match folder naming.

In [ ]:
df = pd.read_csv(val_csv)
df.columns = df.columns.str.strip()

print(df.head())
print("Shape:", df.shape)

video_ids = df["video_name"].astype(str).tolist()
labels = df["label"].astype(int).tolist()

video_ids = [vid.zfill(5) for vid in video_ids]

print("Loaded:", len(video_ids), len(labels))

## Dataset Definition

We define a custom PyTorch Dataset to:
- Load video frames from disk
- Select the first 16 frames per video
- Normalize images using ImageNet statistics
- Convert data into the format expected by VideoMAE: (T, C, H, W)

This step prepares raw video data into a format suitable for model inference.

In [ ]:
class DFEWDataset(Dataset):
    def __init__(self, video_ids, labels, clip_root):
        self.video_ids = video_ids
        self.labels = labels
        self.clip_root = clip_root

    def __len__(self):
        return len(self.video_ids)

    def __getitem__(self, idx):
        vid = self.video_ids[idx]
        label = self.labels[idx]

        frame_folder = os.path.join(self.clip_root, vid)

        if not os.path.exists(frame_folder):
            return self.__getitem__((idx + 1) % len(self))

        frame_files = sorted(os.listdir(frame_folder))[:16]

        frames = []
        for f in frame_files:
            img_path = os.path.join(frame_folder, f)
            img = cv2.imread(img_path)

            if img is None:
                continue

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = img / 255.0

            # ImageNet normalization
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            img = (img - mean) / std

            frames.append(img)

        if len(frames) < 16:
            return self.__getitem__((idx + 1) % len(self))

        frames = np.array(frames)

        # CRITICAL FIX: (T, C, H, W)
        frames = np.transpose(frames, (0, 3, 1, 2))

        return {
            "pixel_values": torch.tensor(frames, dtype=torch.float32),
            "labels": label,
            "video_path": frame_folder
        }

## DataLoader

In [ ]:
dataset = DFEWDataset(video_ids, labels, clip_root)
loader = DataLoader(dataset, batch_size=4, shuffle=False)

## Load Trained Model

We load a VideoMAE model pretrained on Kinetics and adapt it to our task (7 emotion classes).

Then, we load our trained weights from a checkpoint.

The model is set to evaluation mode to disable training-specific behaviors such as dropout.

In [ ]:
model = VideoMAEForVideoClassification.from_pretrained(
    "MCG-NJU/videomae-base-finetuned-kinetics",
    num_labels=7,
    ignore_mismatched_sizes=True
)

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint, strict=False)

model.to(device)
model.eval()

print("Model loaded successfully")

## Model Inference

We run the model on the test dataset to obtain predictions.

For each video:
- The model outputs class probabilities
- We select the class with the highest probability as the prediction

We store:
- Predicted labels
- Ground truth labels
- Video paths for later visualization

In [ ]:
all_preds = []
all_labels = []
all_paths = []

with torch.no_grad():
    for batch in tqdm(loader):
        inputs = batch["pixel_values"].to(device)
        labels_batch = torch.tensor(batch["labels"]).to(device)

        outputs = model(inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels_batch.cpu().numpy())
        all_paths.extend(batch["video_path"])

## Compare Predictions

In [ ]:
correct = []
incorrect = []

for i in range(len(all_preds)):
    if all_preds[i] == all_labels[i]:
        correct.append(i)
    else:
        incorrect.append(i)

print("Correct:", len(correct))
print("Incorrect:", len(incorrect))

## Extract Representative Frame

We define a helper function to extract the middle frame of each video.

This frame serves as a representative snapshot of the video for visualization purposes.

In [ ]:
def get_middle_frame(folder):
    files = sorted(os.listdir(folder))
    mid = len(files) // 2
    img = cv2.imread(os.path.join(folder, files[mid]))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

## Qualitative Results Visualization

We visualize:
- Top row: Correct predictions
- Bottom row: Incorrect predictions

Each image displays:
- Ground Truth (GT)
- Model Prediction (Pred)

This qualitative analysis helps us understand:
- What types of expressions the model recognizes well
- Where it struggles (e.g., subtle or ambiguous emotions)

The figure is saved as `qualitative_examples.png` for use in the final report.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    idx = correct[i]
    img = get_middle_frame(all_paths[idx])

    axes[0, i].imshow(img)
    axes[0, i].set_title(f"GT: {all_labels[idx]} | Pred: {all_preds[idx]}")
    axes[0, i].axis("off")

for i in range(4):
    idx = incorrect[i]
    img = get_middle_frame(all_paths[idx])

    axes[1, i].imshow(img)
    axes[1, i].set_title(f"GT: {all_labels[idx]} | Pred: {all_preds[idx]}")
    axes[1, i].axis("off")

plt.tight_layout()
plt.savefig("qualitative_examples.png")
plt.show()